## Product Clustering

**Goal**
- Group reviews into 4–6 meta categories using embeddings.

In [2]:
!pip install sentence-transformers
import pandas as pd
from pathlib import Path

df = pd.read_csv("../outputs/clean_reviews.csv")

df.head()

,name,reviews.rating,reviews.text,sentiment,clean_review
0,AmazonBasics AAA Performance Alkaline Batterie...,3,I order 3 of them and one of the item is bad q...,neutral,i order 3 of them and one of the item is bad q...
1,AmazonBasics AAA Performance Alkaline Batterie...,4,Bulk is always the less expensive way to go fo...,positive,bulk is always the less expensive way to go fo...
2,AmazonBasics AAA Performance Alkaline Batterie...,5,Well they are not Duracell but for the price i...,positive,well they are not duracell but for the price i...
3,AmazonBasics AAA Performance Alkaline Batterie...,5,Seem to work as well as name brand batteries a...,positive,seem to work as well as name brand batteries a...
4,AmazonBasics AAA Performance Alkaline Batterie...,5,These batteries are very long lasting the pric...,positive,these batteries are very long lasting the pric...


## Prepare Text Data
- Using cleaned reviews as input for clustering.

In [3]:
#extract cleaned review text
texts = df["clean_review"].tolist()

#check number of reviews
len(texts)

59900

## Use Sample for Fast Experiment
- Starting with a smaller sample to test clustering quickly.

In [4]:
#use a sample first to avoid slow execution
sample_df = df.sample(n=1000, random_state=42).copy()

#extract sample texts
sample_texts = sample_df["clean_review"].tolist()

#check sample size
len(sample_texts)

1000

## Create Text Embeddings
- Converting reviews into numerical vectors using a pretrained sentence transformer.

In [ ]:
from sentence_transformers import SentenceTransformer

#load embedding model (fast + good for clustering)
model = SentenceTransformer("all-MiniLM-L6-v2")

#create embeddings for sample reviews
embeddings = model.encode(sample_texts, show_progress_bar=True)

#check shape (rows = reviews, columns = vector size)
embeddings.shape

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

(1000, 384)

: 

## Apply KMeans Clustering
- Grouping reviews into 4–6 clusters.

In [ ]:
from sklearn.cluster import KMeans

# define number of clusters (start with 5)
k = 5

# create and train KMeans
kmeans = KMeans(n_clusters=k, random_state=42)

# assign cluster to each review
sample_df["cluster"] = kmeans.fit_predict(embeddings)

# check results
sample_df[["clean_review", "cluster"]].head()

## Inspect Cluster Sizes
- Checking how many reviews belong to each cluster.

In [ ]:
# count reviews per cluster
sample_df["cluster"].value_counts().sort_index()

## Inspect Cluster Content
- Look at sample reviews from each cluster to understand what they represent.

In [ ]:
#show 5 example reviews per cluster
for cluster_id in sorted(sample_df["cluster"].unique()):
    print(f"\n--- Cluster {cluster_id} ---\n")
    
    examples = sample_df[sample_df["cluster"] == cluster_id]["clean_review"].head(5)
    
    for review in examples:
        print("-", review[:150])  # print first 150 chars 

## Assign Meaningful Names to Clusters
- Based on the example reviews, giving each cluster a human-readable category name.

In [ ]:
#assign names manually based on what you observed
cluster_names = {
    0: "Batteries / Power",
    1: "Electronics / Devices",
    2: "Accessories",
    3: "Home / Everyday Products",
    4: "Other"
}

#map cluster ids to names
sample_df["cluster_name"] = sample_df["cluster"].map(cluster_names)

#check result
sample_df[["clean_review", "cluster", "cluster_name"]].head()

## Save Clustered Sample
- Store clustering results for later use (e.g. summarization).

In [ ]:
#define output path
output_path = Path("../outputs/clustered_sample.csv")

#save clustered data
sample_df.to_csv(output_path, index=False)

print("Saved to:", output_path)